# Codifications 1 d'avril à août : corrections avec la codification anticipée

*Journal : [`tableau_resultats.md`](tableau_resultats.md).*

**Codification anticipée** (notebooks `04` à `08`) : la codification `PAY_n` du mois M porte sur **la facture du mois M** (`BILL_AMTn`, solde de fin de mois), **réglée le mois suivant** par `PAY_AMT(n-1)`. Pour trancher une codification 1 du mois M, on lit donc le trio : codification `PAY_n`, facture du mois `BILL_AMTn`, paiement du mois suivant `PAY_AMT(n-1)`.

**Objet** : les 34 codifications 1 d'avril à août (28 clients, tous codifiés 1 aussi en septembre). Septembre (`PAY_1`) n'est pas traité : le paiement d'octobre est inconnu. `dpnm` n'est utilisé à aucun moment.

**Étapes** :
0. **règle de l'utilisateur** : `PAY_n` = 1 et solde du mois suivant nul ou créditeur (`BILL_AMT(n-1)` <= 0) → **la codification saine antérieure** ;
1. **règle de l'encours nul** : une codification 1 sur un mois **sans facture** (`BILL_AMTn` <= 0) ne peut pas être un retard ; si le mois précédent est sain, le 1 reprend sa codification (règle d'août des premiers essais, généralisée à tous les mois) ;
2. **règle fixe des impayés** (passage 18 du journal, abandonnée depuis dans le projet) : une facture du mois d'au moins 100 NT$ sans aucun paiement le mois suivant vaut un retard (2, ou niveau du mois précédent + 1) ;
3. **décisions à la main** pour les codifications 1 qui restent, avec les repères du `08` : sous la codification anticipée, un retard sort du retard dans 0,2 % des cas sans paiement, 2 % entre 3 et 5 % payés, 39 % de 5 à 10 %, 51 % de 10 à 20 %, 58 % de 20 à 50 %, 53 % de 50 à 90 %, 98 % à 90 % et plus ; un client sain qui paie 3 % ou plus reste sain dans la grande majorité des cas.

## 1. Données

In [8]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())
df = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv").drop(columns="dpnm")   # jamais utilisé
PAY = [f"PAY_{n}" for n in range(1, 7)]
MOIS = {1: "septembre", 2: "août", 3: "juillet", 4: "juin", 5: "mai", 6: "avril"}
origine = df[PAY].copy()   # codifications d'origine, pour les contrôles
print("Codifications 1 par mois :", {MOIS[n]: int((df[f"PAY_{n}"] == 1).sum()) for n in range(1, 7)})

# Colonnes groupées selon la codification anticipée : pour chaque mois, codification, facture du mois, paiement du mois suivant
COLONNES_B = ["ID", "LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"]
for n in range(1, 7):
    COLONNES_B += [f"PAY_{n}", f"BILL_AMT{n}"] + ([f"PAY_AMT{n - 1}"] if n >= 2 else [])
COLONNES_B.append("PAY_AMT6")   # paiement d'avril : règle la facture de mars, hors période

def surligner_1(ligne):
    return ["background-color: #ffe08a; font-weight: bold" if c in PAY and v == 1 else "" for c, v in ligne.items()]

def afficher(table, titre):
    display(Markdown(titre))
    display(table[COLONNES_B].set_index("ID").style.apply(surligner_1, axis=1)
            .format("{:,.0f}", subset=[c for c in COLONNES_B if c.startswith(("BILL_AMT", "PAY_AMT", "LIMIT_BAL"))], thousands=" "))

def detail_1(d):
    # Une ligne par codification 1 d'avril à août : le trio de la codification anticipée et ses voisins
    lignes = []
    for n in range(2, 7):
        for i, r in d[d[f"PAY_{n}"] == 1].iterrows():
            f, p = r[f"BILL_AMT{n}"], r[f"PAY_AMT{n - 1}"]
            lignes.append({"ID": int(r["ID"]), "Mois M": MOIS[n], "Colonne": f"PAY_{n}",
                           "Codification du mois précédent": int(r[f"PAY_{n + 1}"]) if n < 6 else None,
                           "Facture du mois (BILL_AMTn)": int(f), "Paiement le mois suivant (PAY_AMT(n-1))": int(p),
                           "Part payée (%)": round(p / f * 100, 1) if f > 0 else None,
                           "Codification du mois suivant": int(r[f"PAY_{n - 1}"])})
    return pd.DataFrame(lignes)

Codifications 1 par mois : {'septembre': 3688, 'août': 28, 'juillet': 4, 'juin': 2, 'mai': 0, 'avril': 0}


## 2. Les codifications 1 d'avril à août, lues avec la codification anticipée

Chaque client est affiché en entier (sans `dpnm`), colonnes groupées selon la codification anticipée : pour chaque mois, de septembre à avril, la codification `PAY_n`, la facture du mois `BILL_AMTn` et le paiement du mois suivant `PAY_AMT(n-1)` (septembre : paiement d'octobre inconnu ; la dernière colonne, `PAY_AMT6`, règle la facture de mars, hors période). Les codifications 1 sont surlignées. Le second tableau donne, pour chaque codification 1, son trio et la part de la facture payée le mois suivant.

In [9]:
clients = df[(df[PAY[1:]] == 1).any(axis=1)]
afficher(clients.sort_values("ID"), f"**Clients avec une codification 1 d'avril à août : {len(clients)}**")
display(detail_1(clients).set_index(["ID", "Mois M"]))

**Clients avec une codification 1 d'avril à août : 28**

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,BILL_AMT1,PAY_2,BILL_AMT2,PAY_AMT1,PAY_3,BILL_AMT3,PAY_AMT2,PAY_4,BILL_AMT4,PAY_AMT3,PAY_5,BILL_AMT5,PAY_AMT4,PAY_6,BILL_AMT6,PAY_AMT5,PAY_AMT6
ID,,,,,,,,,,,,,,,,,,,,,,,
892,400 000,1,2,2,37,1,13 611,1,15 066,3 019,1,15 882,5 014,2,12 599,3,0,18 599,7 040,0,12 284,2 006,2 005
1967,90 000,1,2,1,29,1,41 073,1,43 480,3 100,1,73 960,1 000,0,2 442,770,0,48 063,48 012,0,48 186,1 756,1 894
5645,500 000,1,1,1,42,1,9 615,1,9 996,1 054,0,177 741,177 080,0,163 299,4 425,0,159 110,5 666,0,135 658,4 825,100 225
6025,210 000,2,2,2,29,1,1 767,1,-3 355,0,-2,-3 987,0,-2,-1 987,2 000,-1,1 697,4 000,0,4 347,3 000,5 000
6118,80 000,1,1,2,26,1,74 384,1,0,5 000,-2,0,0,-2,0,0,-2,0,0,-2,0,0,0
6783,500 000,1,1,1,32,1,48 169,1,53 197,20 000,1,63 010,20 012,1,64 178,10 036,0,64 992,15 000,0,50 722,10 000,50 000
8616,270 000,2,2,2,53,1,716,1,-1 524,2,-2,-3 764,2,-2,-4,6 002,-2,-19 205,2,-1,42 418,64 446,5 002
9120,50 000,1,2,2,40,1,50 209,1,-2 522,0,-1,409,2 931,-1,29 148,30 000,0,30 270,5 270,0,22 814,2 814,3 089
11013,20 000,2,2,2,45,1,4 516,1,-630,700,-1,18 976,21 400,0,8 005,1 005,-1,3 034,3 034,-1,9 287,9 287,85


,,Colonne,Codification du mois précédent,Facture du mois (BILL_AMTn),Paiement le mois suivant (PAY_AMT(n-1)),Part payée (%),Codification du mois suivant
ID,Mois M,,,,,,
892,août,PAY_2,1,15066,3019,20.0,1
1967,août,PAY_2,1,43480,3100,7.1,1
5645,août,PAY_2,0,9996,1054,10.5,1
6025,août,PAY_2,-2,-3355,0,NaN,1
6118,août,PAY_2,-2,0,5000,NaN,1
6783,août,PAY_2,1,53197,20000,37.6,1
8616,août,PAY_2,-2,-1524,2,NaN,1
9120,août,PAY_2,-1,-2522,0,NaN,1
11013,août,PAY_2,-1,-630,700,NaN,1


## 2 bis. Règle de l'utilisateur : pas de solde le mois suivant

Une codification 1 du mois M (`PAY_n` = 1) dont le solde du mois suivant est nul ou créditeur (`BILL_AMT(n-1)` <= 0) **reprend la codification saine antérieure** : la plus récente codification saine (-2, -1 ou 0) avant M (`PAY_(n+1)`, puis `PAY_(n+2)`…). Sans codification saine antérieure (par exemple en avril, premier mois observé), le 1 reste à trancher. Avril à août (n = 2 à 6), traités dans l'ordre ; septembre n'est pas traité (`BILL_AMT0` n'existe pas).

In [10]:
def saine_anterieure(ligne, n):
    # Plus récente codification saine (-2, -1 ou 0) avant le mois n : PAY_(n+1), PAY_(n+2)... ; None s'il n'y en a pas
    for k in range(n + 1, 7):
        if ligne[f"PAY_{k}"] <= 0:
            return ligne[f"PAY_{k}"]
    return None

corriges_mois_suivant, sans_anterieure = [], []
for n in range(6, 1, -1):                      # d'avril à août : un mois corrigé sert d'antériorité au suivant
    regle = (df[f"PAY_{n}"] == 1) & (df[f"BILL_AMT{n - 1}"] <= 0)
    for k in df.index[regle]:
        valeur = saine_anterieure(df.loc[k], n)
        if valeur is None:
            sans_anterieure.append((int(df.at[k, "ID"]), MOIS[n]))
            continue
        df.at[k, f"PAY_{n}"] = valeur
        corriges_mois_suivant.append((int(df.at[k, "ID"]), MOIS[n], int(valeur)))
print(f"Codifications 1 corrigées (solde du mois suivant nul ou créditeur) : {len(corriges_mois_suivant)}")
print("Détail (ID, mois, codification reprise) :", corriges_mois_suivant)
print("Sans codification saine antérieure, restent à trancher :", sans_anterieure)
print("Codifications 1 restantes d'avril à août :", int((df[PAY[1:]] == 1).sum().sum()))

Codifications 1 corrigées (solde du mois suivant nul ou créditeur) : 0
Détail (ID, mois, codification reprise) : []
Sans codification saine antérieure, restent à trancher : []
Codifications 1 restantes d'avril à août : 34


## 3. Règle de l'encours nul

Une codification 1 du mois M **sans facture du mois** (`BILL_AMTn` <= 0) ne peut pas être un retard : il n'y a rien à régler. Si le mois précédent est sain (`PAY_(n+1)` <= 0), le 1 reprend cette codification. Traitement d'avril à août, dans l'ordre (un 1 corrigé sert de mois précédent au mois suivant).

In [11]:
corriges_encours = []
for n in range(5, 1, -1):          # mai (5) à août (2) : avril n'a pas de mois précédent observé
    regle = (df[f"PAY_{n}"] == 1) & (df[f"BILL_AMT{n}"] <= 0) & (df[f"PAY_{n + 1}"] <= 0)
    corriges_encours += [(int(i), n, int(df.at[k, f"PAY_{n + 1}"])) for k, i in df.loc[regle, "ID"].items()]
    df.loc[regle, f"PAY_{n}"] = df.loc[regle, f"PAY_{n + 1}"]
print(f"Codifications 1 corrigées par la règle de l'encours nul : {len(corriges_encours)}")
print("Codifications 1 restantes d'avril à août :", int((df[PAY[1:]] == 1).sum().sum()))

Codifications 1 corrigées par la règle de l'encours nul : 21
Codifications 1 restantes d'avril à août : 13


## 4. Règle fixe des impayés

La règle fixe des impayés (passage 18 du journal), pour les codifications 1 qui restent : une facture du mois d'au moins 100 NT$ sans aucun paiement le mois suivant vaut un retard (2 si le mois précédent est sain, niveau du mois précédent + 1 sinon).

In [12]:
corriges_impayes = []
for n in range(6, 1, -1):
    regle = (df[f"PAY_{n}"] == 1) & (df[f"BILL_AMT{n}"] >= 100) & (df[f"PAY_AMT{n - 1}"] == 0)
    avant = df[f"PAY_{n + 1}"] if n < 6 else pd.Series(np.nan, index=df.index)
    niveau = pd.Series(np.where(avant >= 2, avant + 1, 2), index=df.index)
    corriges_impayes += [(int(i), n) for i in df.loc[regle, "ID"]]
    df.loc[regle, f"PAY_{n}"] = niveau[regle]
print(f"Codifications 1 corrigées par la règle fixe des impayés : {len(corriges_impayes)}")
restants = df[(df[PAY[1:]] == 1).any(axis=1)]
print("Codifications 1 restantes d'avril à août :", int((df[PAY[1:]] == 1).sum().sum()), "chez", len(restants), "clients")

Codifications 1 corrigées par la règle fixe des impayés : 0
Codifications 1 restantes d'avril à août : 13 chez 7 clients


## 5. Les codifications 1 qui restent : à trancher à la main

Les clients dont il reste une codification 1 d'avril à août, affichés en entier (colonnes groupées selon la codification anticipée), puis le trio de chaque codification 1.

In [13]:
if len(restants):
    afficher(restants.sort_values("ID"), f"**Clients à trancher : {len(restants)}**")
    display(detail_1(restants).set_index(["ID", "Mois M"]))

**Clients à trancher : 7**

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,BILL_AMT1,PAY_2,BILL_AMT2,PAY_AMT1,PAY_3,BILL_AMT3,PAY_AMT2,PAY_4,BILL_AMT4,PAY_AMT3,PAY_5,BILL_AMT5,PAY_AMT4,PAY_6,BILL_AMT6,PAY_AMT5,PAY_AMT6
ID,,,,,,,,,,,,,,,,,,,,,,,
892,400 000,1,2,2,37,1,13 611,1,15 066,3 019,1,15 882,5 014,2,12 599,3,0,18 599,7 040,0,12 284,2 006,2 005
1967,90 000,1,2,1,29,1,41 073,1,43 480,3 100,1,73 960,1 000,0,2 442,770,0,48 063,48 012,0,48 186,1 756,1 894
5645,500 000,1,1,1,42,1,9 615,1,9 996,1 054,0,177 741,177 080,0,163 299,4 425,0,159 110,5 666,0,135 658,4 825,100 225
6783,500 000,1,1,1,32,1,48 169,1,53 197,20 000,1,63 010,20 012,1,64 178,10 036,0,64 992,15 000,0,50 722,10 000,50 000
11498,250 000,1,1,1,42,1,352 016,1,341 600,15 210,1,321 232,8 273,1,307 354,8 100,2,285 901,100,2,272 322,8 942,254 000
19195,380 000,2,2,1,33,1,363 944,1,385 726,29 540,2,368 318,74,2,354 443,14 301,2,354 765,13 000,2,354 579,12 800,12 800
25176,50 000,2,2,1,22,1,44 531,1,50 863,7 178,2,49 555,0,2,34 644,1 656,2,28 867,944,2,28 983,967,1 112


,,Colonne,Codification du mois précédent,Facture du mois (BILL_AMTn),Paiement le mois suivant (PAY_AMT(n-1)),Part payée (%),Codification du mois suivant
ID,Mois M,,,,,,
892,août,PAY_2,1,15066,3019,20.0,1
1967,août,PAY_2,1,43480,3100,7.1,1
5645,août,PAY_2,0,9996,1054,10.5,1
6783,août,PAY_2,1,53197,20000,37.6,1
11498,août,PAY_2,1,341600,15210,4.5,1
19195,août,PAY_2,2,385726,29540,7.7,1
25176,août,PAY_2,2,50863,7178,14.1,1
892,juillet,PAY_3,2,15882,5014,31.6,1
1967,juillet,PAY_3,0,73960,1000,1.4,1


## 6. Décisions à la main

Une entrée par client : `ID: {n: nouvelle codification de PAY_n}`, pour n = 2 à 6, avec la raison en commentaire, tirée du trio de la codification anticipée (facture du mois, paiement le mois suivant).

In [14]:
# ID : {n : nouvelle valeur de PAY_n}, pour n = 2 à 6 (codification anticipée)
corrections = {
    # Décisions de l'utilisateur (codification anticipée : facture du mois, part payée le mois suivant)
    6783: {4: 0, 3: 0, 2: 0},    # juin, juillet, août : paie 16 %, 32 %, 38 % de sa facture le mois suivant
    11498: {4: 2, 3: 2, 2: 2},   # en retard en avril et mai ; paie 3 %, 3 %, 4 % de factures au-dessus de son plafond
    892: {3: 0, 2: 0},           # juillet, août : paie 32 %, 20 % de sa facture le mois suivant (juin à 2 laissé tel quel)
    1967: {3: 0, 2: 0},          # juillet, août : paie 1 %, 7 % ; solde en baisse de 73 960 à 43 480 sans paiement correspondant
    5645: {2: 0},                # août : paie 11 % de sa facture (9 996) en septembre
    19195: {2: 2},               # en retard d'avril à juillet ; août : paie 8 % d'une facture au-dessus de son plafond
    25176: {2: 2},               # en retard d'avril à juillet ; août : paie 14 % de sa facture en septembre
}

for id_client, valeurs in corrections.items():
    ligne = df["ID"] == id_client
    assert ligne.any(), f"client {id_client} absent"
    assert all(2 <= n <= 6 for n in valeurs), f"client {id_client} : seules PAY_2 à PAY_6 se corrigent ici"
    assert (origine.loc[ligne, PAY[1:]] == 1).any(axis=1).all(), f"client {id_client} : aucune codification 1 d'avril à août à l'origine"
    avant = df.loc[ligne].copy()
    for n, valeur in valeurs.items():
        df.loc[ligne, f"PAY_{n}"] = valeur
    afficher(pd.concat([avant, df.loc[ligne]]).assign(ID=lambda t: [f"{id_client} avant", f"{id_client} après"]), f"**Client {id_client}**")

print(f"Clients corrigés à la main : {len(corrections)} ; codifications 1 restantes d'avril à août : {int((df[PAY[1:]] == 1).sum().sum())}")

**Client 6783**

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,BILL_AMT1,PAY_2,BILL_AMT2,PAY_AMT1,PAY_3,BILL_AMT3,PAY_AMT2,PAY_4,BILL_AMT4,PAY_AMT3,PAY_5,BILL_AMT5,PAY_AMT4,PAY_6,BILL_AMT6,PAY_AMT5,PAY_AMT6
ID,,,,,,,,,,,,,,,,,,,,,,,
6783 avant,500 000,1,1,1,32,1,48 169,1,53 197,20 000,1,63 010,20 012,1,64 178,10 036,0,64 992,15 000,0,50 722,10 000,50 000
6783 après,500 000,1,1,1,32,1,48 169,0,53 197,20 000,0,63 010,20 012,0,64 178,10 036,0,64 992,15 000,0,50 722,10 000,50 000


**Client 11498**

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,BILL_AMT1,PAY_2,BILL_AMT2,PAY_AMT1,PAY_3,BILL_AMT3,PAY_AMT2,PAY_4,BILL_AMT4,PAY_AMT3,PAY_5,BILL_AMT5,PAY_AMT4,PAY_6,BILL_AMT6,PAY_AMT5,PAY_AMT6
ID,,,,,,,,,,,,,,,,,,,,,,,
11498 avant,250 000,1,1,1,42,1,352 016,1,341 600,15 210,1,321 232,8 273,1,307 354,8 100,2,285 901,100,2,272 322,8 942,254 000
11498 après,250 000,1,1,1,42,1,352 016,2,341 600,15 210,2,321 232,8 273,2,307 354,8 100,2,285 901,100,2,272 322,8 942,254 000


**Client 892**

...

**Client 19195**

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,BILL_AMT1,PAY_2,BILL_AMT2,PAY_AMT1,PAY_3,BILL_AMT3,PAY_AMT2,PAY_4,BILL_AMT4,PAY_AMT3,PAY_5,BILL_AMT5,PAY_AMT4,PAY_6,BILL_AMT6,PAY_AMT5,PAY_AMT6
ID,,,,,,,,,,,,,,,,,,,,,,,
19195 avant,380 000,2,2,1,33,1,363 944,1,385 726,29 540,2,368 318,74,2,354 443,14 301,2,354 765,13 000,2,354 579,12 800,12 800
19195 après,380 000,2,2,1,33,1,363 944,2,385 726,29 540,2,368 318,74,2,354 443,14 301,2,354 765,13 000,2,354 579,12 800,12 800


**Client 25176**

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,BILL_AMT1,PAY_2,BILL_AMT2,PAY_AMT1,PAY_3,BILL_AMT3,PAY_AMT2,PAY_4,BILL_AMT4,PAY_AMT3,PAY_5,BILL_AMT5,PAY_AMT4,PAY_6,BILL_AMT6,PAY_AMT5,PAY_AMT6
ID,,,,,,,,,,,,,,,,,,,,,,,
25176 avant,50 000,2,2,1,22,1,44 531,1,50 863,7 178,2,49 555,0,2,34 644,1 656,2,28 867,944,2,28 983,967,1 112
25176 après,50 000,2,2,1,22,1,44 531,2,50 863,7 178,2,49 555,0,2,34 644,1 656,2,28 867,944,2,28 983,967,1 112


Clients corrigés à la main : 7 ; codifications 1 restantes d'avril à août : 0


**Comment lire** : chaque client corrigé est affiché avant et après. Le compte final doit tomber à 0 quand toutes les codifications 1 d'avril à août sont tranchées.